# Phase 1 baselines on Kaggle

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Settings → **Internet on**.

Results go to `/kaggle/working/outputs`. A run whose `results.json` already exists, either there or in any attached
input under `/kaggle/input/*/outputs`, is **skipped instead of retrained**. To reuse this notebook's results later,
open the later notebook and choose *Add Input → Your Work → this notebook*.

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2, then run again. (On CPU every run takes hours.)"


In [ ]:
# Upgrade transformers only if Kaggle's copy is too old for the training script.
from packaging import version
if version.parse(transformers.__version__) < version.parse("4.46"):
    !pip install -q "transformers>=4.46" "accelerate>=0.26"
    print("Upgraded transformers - restart the kernel (Run -> Restart) and run from the top.")

In [ ]:
# Clone (or update) the project code. If the repo is private, add a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets) holding a GitHub personal access token with read access.
import os, subprocess
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
url = f"https://github.com/{REPO}.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    pass  # public repo: no token needed
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, CODE_DIR], check=True)
%cd {CODE_DIR}

In [ ]:
# Use one GPU so every run has the same effective batch size (2 GPUs would double it).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OUTPUT_DIR"] = "/kaggle/working/outputs"

# Earlier results attached as inputs (e.g. this notebook's previous version) are reused, not retrained.
import glob
prev = sorted(set(glob.glob("/kaggle/input/*/outputs") + glob.glob("/kaggle/input/*/*/outputs")))
os.environ["PREV_RESULTS"] = " ".join(prev)
print("Reusing results from:", prev or "nothing yet")

In [ ]:
# Phase 1: RoBERTa, reply-only vs parent+reply, 5 seeds each (finished runs are skipped).
!bash run_baselines.sh roberta-base

In [ ]:
# Phase 1b (optional): learning-rate tuning, chosen on validation weighted F1 (the paper's metric).
# Searches lr 1e-5 / 2e-5 / 3e-5 on 2 seeds, then runs the best lr on all 5 seeds. ~1-1.25 h for roberta-base.
# !test -d data/counter_context || git clone --depth 1 https://github.com/xinchenyu/counter_context data/counter_context
# !python tune.py --model roberta-base --select_metric weighted_f1 --output_dir /kaggle/working/outputs --prev_results {" ".join(prev)}

In [ ]:
# Phase 2 (optional): stronger encoder. Uncomment to run.
# !bash run_baselines.sh microsoft/deberta-v3-base

In [ ]:
# Final table over everything in /kaggle/working/outputs, also saved as a markdown file.
!python aggregate.py --output_dir /kaggle/working/outputs | tee /kaggle/working/outputs/summary.md